In [35]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, Add, Lambda
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import r2_score
import pandas as pd

In [ ]:
train_data = np.load("data/train_data.npz")
target_scaler = joblib.load("data/target_scaler.pkl")
X_train = train_data['X_train']
X_test  = train_data['X_test']
y_train = train_data['y_train']
y_test  = train_data['y_test']

print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print(y_train.shape)
print(y_train.ndim)

X_train shape: (672, 60, 5)
y_train shape: (672,)
(672,)
1


In [37]:
import random
random.seed(37)
np.random.seed(37)
tf.random.set_seed(37)
#set random seeds for reproducibility

In [38]:
#Positional 
def positional_encoding(seq_len, d_model):
    positions = np.arange(seq_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(1000, (2 * (dims // 2)) / np.float64(d_model))
    angle_rads = positions * angle_rates
    angle_rads[: , 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[: , 1::2] = np.cos(angle_rads[:, 1::2])
    return tf.cast(angle_rads[np.newaxis, ...], dtype = tf.float32)

In [ ]:
def transformer_encoder(inputs, num_heads, key_dim, ff_dim, dropout):
    x = LayerNormalization(epsilon = 1e-6)(inputs)
    attention = MultiHeadAttention(num_heads = num_heads, key_dim = key_dim)(x, x)
    attention = Dropout(dropout)(attention)
    x = Add()([inputs, attention])
    ff = LayerNormalization(epsilon = 1e-6)(x)
    ff = Dense(ff_dim, activation = "gelu")(ff)
    ff = Dropout(dropout)(ff)
    ff = Dense(inputs.shape[-1])(ff)
    ff = Dropout(dropout)(ff)
    output = Add()([x, ff])
    return output

In [ ]:
seq_len = X_train.shape[1]
n_features = X_train.shape[2]
d_model = 64
num_heads = 4
ff_dim = 128
dropout = 0.2

input = Input(shape = (seq_len, n_features))
x = Dense(d_model)(input)
pos_encoding = tf.cast(positional_encoding(seq_len, d_model), dtype=tf.float32)
x = x + pos_encoding
x = transformer_encoder(x, num_heads = num_heads, key_dim = d_model // num_heads, ff_dim = ff_dim, dropout = dropout)
x = Lambda(lambda t: t[:, -1, :])(x)
x = Dense(64, activation = "gelu")(x)
x = Dropout(dropout)(x)
x = Dense(32, activation = "gelu")(x)
output = Dense(y_train.shape[1], activation = "softplus")(x)
model = Model(inputs = input, outputs = output)
model.compile(optimizer = Adam(learning_rate = 1e-4), loss = "mse")
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 60, 5)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 60, 64)    │        384 │ input_layer_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 60, 64)    │          0 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 60, 64)    │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 60, 64)    │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 60, 64)    │          0 │ add_10[0][0],     │
│                     │                   │            │ dropout_16[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 60, 64)    │        128 │ add_11[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 60, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 60, 128)   │          0 │ dense_17[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 60, 64)    │      8,256 │ dropout_17[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 60, 64)    │          0 │ dense_18[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (None, 60, 64)    │          0 │ add_11[0][0],     │
│                     │                   │            │ dropout_18[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_3 (Lambda)   │ (None, 64)        │          0 │ add_12[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 64)        │      4,160 │ lambda_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 64)        │          0 │ dense_19[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 32)        │      2,080 │ dropout_19[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 1)         │         33 │ dense_20[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 40,129 (156.75 KB)

 Trainable params: 40,129 (156.75 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
from tensorflow.python.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor='val_loss', patience=10, min_delta=1e-5, restore_best_weights=True)
#early stopping to prevent overfitting and optimize training time
history = model.fit(X_train, y_train, epochs=150, batch_size=32, validation_split=0.2, callbacks=[es])

Epoch 1/150
17/17 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - loss: 0.0846 - val_loss: 0.0277
Epoch 2/150
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 0.0552 - val_loss: 0.0132
Epoch 3/150
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 0.0449 - val_loss: 0.0067
Epoch 4/150
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 0.0414 - val_loss: 0.0091
Epoch 5/150
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 0.0314 - val_loss: 0.0051
Epoch 6/150
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 0.0283 - val_loss: 0.0044
Epoch 7/150
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 0.0197 - val_loss: 0.0042
Epoch 8/150
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 0.0195 - val_loss: 0.0038
Epoch 9/150
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - loss: 0.0162 - val_loss: 0.0042
Epoch 10/150
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 0.0154 - val_loss: 0.0038
Epoch 11/150
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 0.0126 - val_loss: 0.0040
Epoch 12/150
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step

In [ ]:
# Cell - Inverse transform
y_actual = target_scaler.inverse_transform(y_test)
y_pred   = target_scaler.inverse_transform(model.predict(X_test))

ValueError: Expected 2D array, got 1D array instead:
array=[0.62799497 0.60650677 0.65483735 0.65681004 0.6532872  0.64497409
 0.61876548 0.60488643 0.61693371 0.6134816  0.62989715 0.63680136
 0.62087921 0.57360538 0.55958519 0.54450846 0.54401518 0.56620777
 0.54352178 0.54091532 0.5280224  0.49991172 0.52506336 0.5273179
 0.4930778  0.50216631 0.49272555 0.52513382 0.52886777 0.53295381
 0.55239902 0.55183544 0.56134655 0.56965999 0.56451681 0.56493958
 0.58325709 0.60608422 0.61608839 0.6113681  0.6261632  0.64004214
 0.61939947 0.62848792 0.63651974 0.67449389 0.70042022 0.68090493
 0.70147697 0.68224364 0.67484581 0.67745665 0.69749594 0.69290935
 0.70793874 0.69650796 0.68832309 0.65960488 0.66263903 0.66609638
 0.64711592 0.65565337 0.65205498 0.63723714 0.64147093 0.67759779
 0.6973548  0.6816196  0.69058091 0.67449323 0.65974591 0.67364639
 0.68860504 0.69142764 0.71160797 0.7056103  0.72254492 0.73581
 0.72557864 0.73334019 0.7426543  0.72875393 0.72430881 0.74632364
 0.7575

In [ ]:
# Cell - Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


mae      = mean_absolute_error(y_actual, y_pred)
rmse     = np.sqrt(mean_squared_error(y_actual, y_pred))
r2       = r2_score(y_actual, y_pred)


print(f'MAE:       {mae:.4f}')
print(f'RMSE:      {rmse:.4f}')
print(f'R²:        {r2:.4f}')

In [ ]:
plt.plot(y_actual[:100, 0], label='Actual')
plt.plot(y_pred[:100, 0], label='Predicted')
plt.legend()
plt.title('LSTM Prediction')
plt.show()
print(y_actual[:5, 0])
print(y_pred[:5, 0])

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Training vs Validation Loss')
plt.show()

In [ ]:
model.save('artifact/lstm_model.h5')